In [1]:
import pandas as pd

from configs import (
    FREQDICT_FILE, ANKIFREQ_FILE,
    RANK_COL, LEMMA_COL, FREQUENCY_COL, POS_COL, WORD_COL,
    MAX_RANK
)

In [2]:
df = pd.read_csv(FREQDICT_FILE)

df

,word,frequency,lemma,morph,pos,rank
0,das,5502675,der,Case=Nom|Definite=Def|Gender=Neut|Number=Sing|...,DET,1
1,dem,5502675,der,Case=Dat|Definite=Def|Gender=Masc|Number=Sing|...,DET,1
2,dessen,5502675,der,Case=Gen|Number=Sing|PronType=Dem,PRON,1
3,der,5502675,der,Case=Nom|Definite=Def|Gender=Masc|Number=Sing|...,DET,1
4,ders,5502675,der,Case=Gen|Number=Sing,PROPN,1
...,...,...,...,...,...,...
1152561,lamik,1,lamik,Case=Nom|Gender=Fem|Number=Sing,NOUN,565801
1152562,lamich,1,lamich,Case=Nom|Gender=Masc|Number=Sing,NOUN,565801
1152563,aaaa,1,aaaa,Case=Nom|Gender=Masc|Number=Sing,NOUN,565801
1152564,ﬂiegenden,1,ﬂiegend,VerbForm=Inf,VERB,565801


In [3]:
df_anki = pd.read_csv(ANKIFREQ_FILE)

df_anki

,word,lemma,frequency,rank
0,Abend,abend,6236,3379
1,Abendessen,abendessen,586,19078
2,Abenteuer,abenteuer,1326,11369
3,Abfahrt,abfahrt,566,19572
4,Abfall,abfall,428,23247
...,...,...,...,...
2658,überrascht,überraschen,2399,7582
2659,übersetzen,übersetzen,1160,12343
2660,überweisen,überweisen,779,16019
2661,üblich,üblich,2694,6925


In [4]:
df_top_rank = df[(df[RANK_COL] < MAX_RANK) & (df[FREQUENCY_COL] > 0)]

total_top_lemmas = df_top_rank[LEMMA_COL].nunique()

total_anki_lemmas = df_anki[LEMMA_COL].nunique()

In [5]:
anki_top_lemma_filter = df_anki[LEMMA_COL].isin(df_top_rank[LEMMA_COL].unique())

anki_lemmas_in_top = df_anki[anki_top_lemma_filter][LEMMA_COL].unique()
anki_lemmas_notin_top = df_anki[~anki_top_lemma_filter][LEMMA_COL].unique()
top_lemmas_missingin_anki = set(df_top_rank[LEMMA_COL].unique()) - set(anki_lemmas_in_top)

In [6]:
print("Summary of initial lemma comparison between Anki and top frequency list:")
print(f"Total top lemmas: {total_top_lemmas}")
print(f"Total Anki lemmas: {total_anki_lemmas}")
print(f"Total Anki lemmas available in top lemmas: {len(anki_lemmas_in_top)}")
print(f"Total Anki lemmas not available in top lemmas: {len(anki_lemmas_notin_top)}")
print(f"Total top lemmas missing from Anki: {len(top_lemmas_missingin_anki)}")

Summary of initial lemma comparison between Anki and top frequency list:
Total top lemmas: 2587
Total Anki lemmas: 2573
Total Anki lemmas available in top lemmas: 1176
Total Anki lemmas not available in top lemmas: 1397
Total top lemmas missing from Anki: 1411


In [7]:
anki_keep = df_anki[df_anki[LEMMA_COL].isin(anki_lemmas_in_top)]
anki_remove = df_anki[df_anki[LEMMA_COL].isin(anki_lemmas_notin_top)]
anki_add = df_top_rank[df_top_rank[LEMMA_COL].isin(top_lemmas_missingin_anki)]

# Filtering by POS

In [8]:
anki_add.groupby(POS_COL)[LEMMA_COL].nunique()

pos
ADJ      382
ADP       16
ADV      366
AUX        9
CCONJ      6
DET       14
NOUN     592
NUM       14
PART       2
PRON      54
PROPN    274
SCONJ      4
VERB     383
X         46
Name: lemma, dtype: int64

In [9]:
def filter_by_pos(df, pos):
    return df[df[POS_COL] == pos].sort_values(by=WORD_COL)

def filter_by_single_pos(df, pos):
    df_local = df.copy()
    df_local['POS_count'] = df_local.groupby(LEMMA_COL)[POS_COL].transform('nunique')
    return df_local[(df_local[POS_COL] == pos) & (df_local['POS_count'] == 1)].sort_values(by=WORD_COL)

## Pos: X

In [10]:
filter_by_single_pos(anki_add, 'X')

,word,frequency,lemma,morph,pos,rank,POS_count
9442,business,1763,business,Foreign=Yes,X,9431,1
9005,campus,1892,campus,Foreign=Yes,X,8999,1
7305,cm,2526,cm,NaN,X,7301,1
4018,cookies,5127,cookies,Foreign=Yes,X,4019,1
8502,corona,2077,corona,Foreign=Yes,X,8503,1
5097,de,3943,de,Foreign=Yes,X,5098,1
9432,dm,1763,dm,NaN,X,9431,1
7946,eur,2282,eur,NaN,X,7947,1
3833,fc,5363,fc,NaN,X,3834,1
6552,ii,2908,ii,NaN,X,6553,1


In [11]:
# Remove all the instances with pos 'X' from anki_add
anki_add = anki_add[anki_add[POS_COL] != 'X']

## Pos: SCONJ

In [12]:
filter_by_single_pos(anki_add, 'SCONJ')

,word,frequency,lemma,morph,pos,rank,POS_count
9415,daß,1768,daß,NaN,SCONJ,9415,1
5428,sofern,3635,sofern,NaN,SCONJ,5429,1
1740,warum,11973,warum,PronType=Int,SCONJ,1741,1
984,wo,20666,wo,PronType=Int,SCONJ,985,1


## Pos: AUX

In [13]:
filter_by_single_pos(anki_add, 'AUX')

,word,frequency,lemma,morph,pos,rank,POS_count
2837,kannst,7289,könnsten,Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbF...,AUX,2838,1
404,muss,45435,mussen,Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbF...,AUX,404,1
403,mussen,45435,mussen,Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbF...,AUX,404,1
2323,musste,8908,musste,Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbF...,AUX,2324,1


In [14]:
# Remove all the instances with pos 'AUX' from anki_add
anki_add = anki_add[anki_add[POS_COL] != 'AUX']

## Pos: DET

In [15]:
filter_by_single_pos(anki_add, 'DET')

,word,frequency,lemma,morph,pos,rank,POS_count
3924,all,5231,all,PronType=Ind,DET,3925,1
7330,eure,2508,eure,Case=Nom|Gender=Fem|Number=Sing|Poss=Yes|PronT...,DET,7331,1
639,mein,31983,mein,Case=Nom|Gender=Neut|Number=Sing|Poss=Yes|Pron...,DET,639,1
638,meine,31983,mein,Case=Nom|Gender=Fem|Number=Sing|Poss=Yes|PronT...,DET,639,1
640,meinem,31983,mein,Case=Dat|Gender=Masc|Number=Sing|Poss=Yes|Pron...,DET,639,1
641,meiner,31983,mein,Case=Dat|Gender=Fem|Number=Sing|Poss=Yes|PronT...,DET,639,1
198,unser,98462,unser,Case=Nom|Gender=Neut|Number=Sing|Poss=Yes|Pron...,DET,197,1
202,unsere,98462,unser,Case=Nom|Gender=Fem|Number=Sing|Poss=Yes|PronT...,DET,197,1
201,unserem,98462,unser,Case=Dat|Gender=Neut|Number=Sing|Poss=Yes|Pron...,DET,197,1
199,unseren,98462,unser,Case=Acc|Gender=Masc|Number=Sing|Poss=Yes|Pron...,DET,197,1


In [16]:
filter_by_pos(anki_add, 'DET')[WORD_COL].unique()

<StringArray>
[                'all',               'allen',               'aller',
 'außergewöhnlichsten',                'dein',               'deine',
              'deinem',              'deinen',              'deiner',
              'deines',           'demselben',           'diejenige',
               'diese',            'dieselbe',           'dieselben',
              'diesem',              'diesen',              'dieses',
                'eure',                'ihre',               'ihrem',
               'ihren',               'ihrer',               'ihres',
                'jene',               'jenem',               'jener',
               'jenes',          'mehrfachem',                'mein',
               'meine',              'meinem',              'meiner',
               'unser',              'unsere',             'unserem',
             'unseren',             'unserer',             'unseres',
              'unserm',              'unsers',              'welche',
      

In [17]:
# Remove all the instances with pos 'DET' from anki_add
anki_add = anki_add[anki_add[POS_COL] != 'DET']

## Pos: ADP

In [18]:
filter_by_single_pos(anki_add, 'ADP')

,word,frequency,lemma,morph,pos,rank,POS_count
5527,angesichts,3535,angesichts,NaN,ADP,5528,1
5229,anhand,3814,anhand,NaN,ADP,5230,1
9342,anlässlich,1785,anlässlich,NaN,ADP,9343,1
1727,aufgrund,12206,aufgrund,NaN,ADP,1728,1
8226,einschließlich,2159,einschließlich,NaN,ADP,8224,1
6691,entlang,2836,entlang,NaN,ADP,6692,1
7460,hinsichtlich,2440,hinsichtlich,NaN,ADP,7453,1
2165,hinter,9444,hinter,NaN,ADP,2165,1
2164,hinterm,9444,hinter,Case=Dat|Gender=Masc|Number=Sing,ADP,2165,1
2166,hinters,9444,hinter,Case=Acc|Gender=Neut|Number=Sing,ADP,2165,1


## Pos: NUM

In [19]:
filter_by_single_pos(anki_add, 'NUM')

,word,frequency,lemma,morph,pos,rank,POS_count
1129,vier,18639,vier,NaN,NUM,1130,1
1907,zehn,10895,zehn,NaN,NUM,1908,1
5140,zwölf,3913,zwölf,NaN,NUM,5141,1


In [20]:
filter_by_pos(anki_add, 'NUM')

,word,frequency,lemma,morph,pos,rank
3161,acht,6672,acht,NaN,NUM,3160
593,drei,34135,drei,NaN,NUM,592
6670,elf,2845,elf,NaN,NUM,6671
1437,fünf,14767,fünf,NaN,NUM,1438
5461,halben,3617,halb,NaN,NUM,5461
6481,hundert,2941,hundert,NaN,NUM,6479
5013,neun,4016,neun,NaN,NUM,5013
2019,sechs,10374,sechs,NaN,NUM,2019
3196,sieben,6614,sieben,NaN,NUM,3197
6117,tausend,3131,tausend,NaN,NUM,6116


In [21]:
# Remove all the lemmas of instances with pos 'NUM' from anki_add, regardless of their pos
anki_add = anki_add[~anki_add[LEMMA_COL].isin(filter_by_pos(anki_add, 'NUM')[LEMMA_COL].unique())]

## Pos: PART

In [22]:
filter_by_single_pos(anki_add, 'PART')

,word,frequency,lemma,morph,pos,rank,POS_count


## Pos: PRON

In [27]:
filter_by_single_pos(anki_add, 'PRON').sort_values(by=LEMMA_COL)

,word,frequency,lemma,morph,pos,rank,POS_count
8006,demjenigen,2255,derjenige,Case=Dat|Gender=Masc|Number=Sing|PronType=Dem,PRON,8001,1
8003,denjenigen,2255,derjenige,Case=Dat|Number=Plur|PronType=Dem,PRON,8001,1
8001,derjenige,2255,derjenige,Case=Nom|Gender=Masc|Number=Sing|PronType=Dem,PRON,8001,1
8004,diejenigen,2255,derjenige,Case=Nom|Number=Plur|PronType=Dem,PRON,8001,1
7319,denselben,2514,derselbe,Case=Dat|Number=Plur|PronType=Dem,PRON,7315,1
7324,derselbe,2514,derselbe,Case=Nom|Gender=Masc|Number=Sing|PronType=Dem,PRON,7315,1
7325,derselben,2514,derselbe,Case=Dat|Number=Plur|PronType=Dem,PRON,7315,1
7326,desselben,2514,derselbe,Case=Gen|Number=Sing|PronType=Dem,PRON,7315,1
2048,dich,10177,dich,Case=Acc|Number=Sing|Person=2|PronType=Prs|Ref...,PRON,2049,1
103,dies,290732,dieser,Case=Nom|Gender=Neut|Number=Sing|PronType=Dem,PRON,100,1


In [ ]:
lemmas_to_keep = ["derjenige", "derselbe", "mehrere", "sämtlicher", "viele", "weniger", "wer"]

# Remove all the lemmas of instances with pos 'PRON' from anki_add, if lemma is not in lemmas_to_keep
anki_add = anki_add[
    (~anki_add[LEMMA_COL].isin(filter_by_pos(anki_add, 'PRON')[LEMMA_COL].unique())) | (anki_add[LEMMA_COL].isin(lemmas_to_keep))]